# 03a — Time-Based Performance Analysis (Q1–Q6)

> **Session 3A: Window functions for temporal analysis**

---

## Setup

In [3]:
# ── Connect to database ─────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from shared_setup import get_connection
conn = get_connection()

Connected to existing database: /home/mohamed/Downloads/ITI_Analytical_SQL_project/ecommerce.db
  Fact_Order_Line: 15,000 rows


---
## Q1 — Cumulative Revenue

Running total revenue over time at monthly granularity

In [38]:
# Q1 — Cumulative Revenue
# TODO: Session 3A

monthly_cumulative_revenue=conn.execute(''' WITH monthly AS (
  SELECT
    date_trunc('month', d.full_date) AS month,
    SUM(f.gross_amount) AS monthly_revenue
  FROM Fact_Order_Line f
  JOIN Dim_Date d ON f.date_key = d.date_key
  GROUP BY 1
)
SELECT
  month,
  SUM(monthly_revenue) OVER (
    ORDER BY month
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
  ) AS cumulative_monthly_revenue
FROM monthly
ORDER BY month;   ''').fetchdf()

print(monthly_cumulative_revenue.head(30))

        month  cumulative_monthly_revenue
0  2022-01-01                   331861.46
1  2022-02-01                   624667.74
2  2022-03-01                   880586.11
3  2022-04-01                  1198782.00
4  2022-05-01                  1503314.58
5  2022-06-01                  1814883.81
6  2022-07-01                  2129861.19
7  2022-08-01                  2527889.61
8  2022-09-01                  2919132.46
9  2022-10-01                  3236720.90
10 2022-11-01                  3695569.92
11 2022-12-01                  4295722.95
12 2023-01-01                  4648742.97
13 2023-02-01                  4993293.24
14 2023-03-01                  5454009.09
15 2023-04-01                  5802246.40
16 2023-05-01                  6262654.05
17 2023-06-01                  6660917.12
18 2023-07-01                  7124223.59
19 2023-08-01                  7539621.24
20 2023-09-01                  7991010.57
21 2023-10-01                  8376253.13
22 2023-11-01                  892

---
## Q2 — Month-to-Date (MTD) Performance

Daily cumulative revenue within each month

In [ ]:
# Q2 — Month-to-Date (MTD) Performance
# TODO: Session 3A

Daily_cumulative_revenue=conn.execute(''' WITH daily AS (
  SELECT
    date_trunc('month', d.full_date) AS month,
    d.full_date as day,
    sum(f.gross_amount) AS daily_revenue
  FROM Fact_Order_Line f
  JOIN Dim_Date d ON f.date_key = d.date_key
  Group by 1,2
  
)
SELECT
  month,
  day,

  SUM(daily_revenue) OVER (
    PARTITION BY MONTH
    ORDER BY day
    
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as Daily_cumulative_revenue
    from daily
    order by month,day
    ''').fetchdf()

print(Daily_cumulative_revenue.head())




       month       date  Daily_cumulative_revenue
0 2022-01-01 2022-01-01                  19667.94
1 2022-01-01 2022-01-02                  30002.84
2 2022-01-01 2022-01-03                  37137.49
3 2022-01-01 2022-01-04                  38514.18
4 2022-01-01 2022-01-05                  41018.26


---
## Q3 — Year-to-Date (YTD) Profit

Cumulative profit from January of each year

In [34]:
# Q3 — Year-to-Date (YTD) Profit
# TODO: Session 3A
monthly_cumulative_profit=conn.execute(''' WITH monthly_prof AS (
  SELECT
    strftime(d.full_date, '%m') AS month,
    strftime(d.full_date, '%Y') AS year,
    SUM(f.profit_amount) AS monthly_profit
     
  FROM Fact_Order_Line f
  JOIN Dim_Date d ON f.date_key = d.date_key
  GROUP BY 1,2
)
SELECT
  month,
  year,
  case 
  when month != '01'   then
  SUM(monthly_profit) OVER (
    partition by year
    ORDER BY year,month
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
  ) 
  else
  monthly_profit
  
  end AS cumulative_monthly_profit
FROM monthly_prof
   ''').fetchdf()

print(monthly_cumulative_profit.head(30))

   month  year  cumulative_monthly_profit
0     01  2022                  102558.11
1     02  2022                  199658.33
2     03  2022                  279504.22
3     04  2022                  379803.95
4     05  2022                  474185.20
5     06  2022                  570506.19
6     07  2022                  674557.59
7     08  2022                  799536.48
8     09  2022                  922844.04
9     10  2022                 1017843.79
10    11  2022                 1171869.91
11    12  2022                 1376479.23
12    01  2025                  147932.58
13    02  2025                  289857.82
14    03  2025                  489289.50
15    04  2025                  698982.51
16    05  2025                  927594.70
17    06  2025                 1135220.45
18    07  2025                 1346014.62
19    08  2025                 1539435.09
20    09  2025                 1732793.83
21    10  2025                 1919360.15
22    11  2025                 218

---
## Q4 — 3-Month Moving Average

Smooth monthly revenue volatility with rolling average

In [43]:
# Q4 — 3-Month Moving Average
# TODO: Session 3A
Three_months_Moving_Average=conn.execute(''' WITH three_months AS (
  SELECT
    strftime(d.full_date, '%m') AS month,
    strftime(d.full_date, '%Y') AS year,
    SUM(f.gross_amount) AS monthly_revenue
  FROM Fact_Order_Line f
  JOIN Dim_Date d ON f.date_key = d.date_key
  GROUP BY 1,2
)
SELECT
  month,
  Year,
  avg(monthly_revenue) OVER (
    ORDER BY year,month
    ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
  ) AS moving_average_Three_months_revenue
FROM three_months
   ''').fetchdf()

print(Three_months_Moving_Average.head(30))

   month  year  moving_average_Three_months_revenue
0     01  2022                        331861.460000
1     02  2022                        312333.870000
2     03  2022                        293528.703333
3     04  2022                        288973.513333
4     05  2022                        292882.280000
5     06  2022                        311432.566667
6     07  2022                        310359.730000
7     08  2022                        341525.010000
8     09  2022                        368082.883333
9     10  2022                        368953.236667
10    11  2022                        389226.770000
11    12  2022                        458863.496667
12    01  2023                        470674.023333
13    02  2023                        432574.440000
14    03  2023                        386095.380000
15    04  2023                        384501.143333
16    05  2023                        423120.270000
17    06  2023                        402302.676667
18    07  20

---
## Q5 — Month-over-Month Comparison

Compare each month with previous using LAG()

In [68]:
# Q5 — Month-over-Month Comparison
# TODO: Session 3A
monthly_revenue_comparison = conn.execute('''
WITH monthly_compare AS (
  SELECT
    strftime(d.full_date, '%m') AS month,
    strftime(d.full_date, '%Y') AS year,
    SUM(f.gross_amount) AS monthly_revenue
  FROM Fact_Order_Line f
  JOIN Dim_Date d ON f.date_key = d.date_key
  GROUP BY 1, 2
),
with_prev AS (
  SELECT
    month,
    year,
    monthly_revenue,
    LAG(monthly_revenue) OVER (ORDER BY year, month) AS prev_month_revenue
  FROM monthly_compare
)
SELECT
  month,
  year,
  monthly_revenue,
  prev_month_revenue,
  CASE
    WHEN prev_month_revenue IS NULL THEN 'first'
    WHEN monthly_revenue > prev_month_revenue THEN 'increasing'
    WHEN monthly_revenue < prev_month_revenue THEN 'decreasing'
    ELSE 'same'
  END AS monthly_comparison
FROM with_prev
ORDER BY year, month;
''').fetchdf()

print(monthly_revenue_comparison.head(30))


   month  year  monthly_revenue  prev_month_revenue monthly_comparison
0     01  2022        331861.46                 NaN              first
1     02  2022        292806.28           331861.46         decreasing
2     03  2022        255918.37           292806.28         decreasing
3     04  2022        318195.89           255918.37         increasing
4     05  2022        304532.58           318195.89         decreasing
5     06  2022        311569.23           304532.58         increasing
6     07  2022        314977.38           311569.23         increasing
7     08  2022        398028.42           314977.38         increasing
8     09  2022        391242.85           398028.42         decreasing
9     10  2022        317588.44           391242.85         decreasing
10    11  2022        458849.02           317588.44         increasing
11    12  2022        600153.03           458849.02         increasing
12    01  2023        353020.02           600153.03         decreasing
13    

---
## Q6 — Revenue Acceleration / Deceleration

Rate of change of the growth rate (second derivative)

In [ ]:
# Q6 — Revenue Acceleration / Deceleration
# TODO: Session 3A

revenue_acceleration = conn.execute('''
WITH monthly AS (
  SELECT
    date_trunc('month', d.full_date) AS month_start,
    strftime(d.full_date, '%Y') AS year,
    strftime(d.full_date, '%m') AS month,
    SUM(f.gross_amount) AS monthly_revenue
  FROM Fact_Order_Line f
  JOIN Dim_Date d ON f.date_key = d.date_key
  GROUP BY 1, 2, 3
),
with_growth AS (
  SELECT
    month_start,
    year,
    month,
    monthly_revenue,
    LAG(monthly_revenue) OVER (ORDER BY month_start) AS prev_month_revenue,
    CASE
      WHEN LAG(monthly_revenue) OVER (ORDER BY month_start) IS NULL THEN NULL
      WHEN LAG(monthly_revenue) OVER (ORDER BY month_start) = 0 THEN NULL
      ELSE (monthly_revenue - LAG(monthly_revenue) OVER (ORDER BY month_start))
           / LAG(monthly_revenue) OVER (ORDER BY month_start)
    END AS growth_rate
  FROM monthly
),
with_accel AS (
  SELECT
    month_start,
    year,
    month,
    monthly_revenue,
    prev_month_revenue,
    growth_rate,
    LAG(growth_rate) OVER (ORDER BY month_start) AS prev_growth_rate,
    growth_rate - LAG(growth_rate) OVER (ORDER BY month_start) AS acceleration
  FROM with_growth
)
SELECT
  year,
  month,
  monthly_revenue,
  prev_month_revenue,
  growth_rate,
  prev_growth_rate,
  acceleration
FROM with_accel
ORDER BY month_start;
''').fetchdf()

print(revenue_acceleration.head())


   year month  monthly_revenue  prev_month_revenue  growth_rate  \
0  2022    01        331861.46                 NaN          NaN   
1  2022    02        292806.28           331861.46    -0.117685   
2  2022    03        255918.37           292806.28    -0.125981   
3  2022    04        318195.89           255918.37     0.243349   
4  2022    05        304532.58           318195.89    -0.042940   

   prev_growth_rate  acceleration  
0               NaN           NaN  
1               NaN           NaN  
2         -0.117685     -0.008295  
3         -0.125981      0.369330  
4          0.243349     -0.286289  


: 